# 11 — Paired Scenario Generation & Counterfactual Construction
**Project:** Causal-History Dependence in Robot Policy Recovery  
**Phase:** 5 (Paired Scenario Generation & Matching-Yield Optimization)  
**Objective:** Demonstrate the paired replay-injection matching strategy, verifying exact physical state matching ($D \le 10^{-6}$) across conditions B0, B1, B2, B3, and B4.

In [1]:
import os, sys, json, yaml
import numpy as np
import matplotlib.pyplot as plt

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(repo_root, "src"))

from causal_history_recovery.environments.push_target_env import PushTargetEnv
from causal_history_recovery.policies.policy_registry import load_policy
from causal_history_recovery.counterfactuals.paired_scenario_generator import PairedScenarioGenerator

cfg_path = os.path.join(repo_root, "configs", "scenarios", "paired_scenario_sampling.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

print("Paired sampling configuration loaded successfully.")
print("Severity tiers:", list(cfg["severity_tiers"].keys()))


Paired sampling configuration loaded successfully.
Severity tiers: ['small', 'medium', 'large']


## 1. Load Policy and Initialize Generator
We load the trained History-Aware GRU policy and initialize the PairedScenarioGenerator.

In [2]:
env = PushTargetEnv()
ckpt_path = os.path.join(repo_root, "models", "checkpoints", "history_gru_best.pt")
policy = load_policy(ckpt_path, device="cpu")
generator = PairedScenarioGenerator(env, config=cfg)

print("Environment and Policy loaded. Ready for paired generation.")


Environment and Policy loaded. Ready for paired generation.


## 2. Sample Single Source Failure & Generate Conditions
We sample a source failure episode under seed 77 with medium severity perturbation.

In [3]:
source = generator.sample_source_failure(
    policy=policy,
    seed=77,
    severity_tier="medium",
    offset_y=0.013,
    perturb_step=25,
    direction=1.0,
    impulse_duration=1,
)

assert source is not None, "Failure not detected!"
print(f"Source Deviation Detected at Step: {source['deviation_step']}")
print(f"Peak Yaw Deflection: {source['peak_yaw_deg']:.2f} deg, Drift: {source['lateral_drift_m']*100:.2f} cm")
print(f"Oracle Recoverable: {source['is_recoverable']}")

scenario = generator.generate_paired_scenario(source, policy, policy_id="history_gru")
print(f"Generated Scenario UID: {scenario['scenario_uid']}")


Source Deviation Detected at Step: 71
Peak Yaw Deflection: 59.78 deg, Drift: 5.20 cm
Oracle Recoverable: False


Generated Scenario UID: scen_history_gru_s77_t71_066db8


## 3. Verify Exact Physical State Equivalence
We verify that all five conditions (B0, B1, B2, B3, B4) achieve state-matching error $D \le 10^{-6}$.

In [4]:
print("Condition-by-Condition Verification:")
print("-" * 75)
for cond_name, c_data in scenario["conditions"].items():
    print(f"Condition {cond_name}: Error = {c_data['state_matching_error']:.10f} | Horizon = {c_data['remaining_horizon']} | Hidden Norm = {c_data['hidden_norm']:.4f} | Obs: {c_data['obs_status']} | Act: {c_data['action_status']}")
    assert c_data["state_matching_error"] <= 1e-6, "State matching error exceeded!"
print("-" * 75)
print("Exact physical state matching by construction: VERIFIED")


Condition-by-Condition Verification:
---------------------------------------------------------------------------
Condition B0: Error = 0.0000000000 | Horizon = 78 | Hidden Norm = 0.0000 | Obs: preserved | Act: preserved
Condition B1: Error = 0.0000000000 | Horizon = 78 | Hidden Norm = 0.0000 | Obs: preserved | Act: ablated
Condition B2: Error = 0.0000000000 | Horizon = 78 | Hidden Norm = 0.0000 | Obs: ablated | Act: preserved
Condition B3: Error = 0.0000000000 | Horizon = 78 | Hidden Norm = 0.0000 | Obs: preserved | Act: preserved
Condition B4: Error = 0.0000000000 | Horizon = 78 | Hidden Norm = 0.0000 | Obs: ablated | Act: ablated
---------------------------------------------------------------------------
Exact physical state matching by construction: VERIFIED


## 4. Interpretation & Synthesis
The paired replay-injection strategy solves the matching scarcity problem by construction: by injecting the exact physical snapshot $s(t_d)$ or replaying the action prefix, physical initial conditions are identical across all counterfactual interventions with bitwise precision.